In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Bronze
# Notebook        : bronze_employees
# Source          : employees.csv
# Target          : procurement.bronze.bronze_employees
# Audit Table     : procurement.audit.duplicate_employees
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads raw employees master data into the Bronze layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Load Employees master data from the source CSV file into the Bronze layer.
#
# Preserve the raw source data with minimal transformations.
#
# Detect duplicate Employee IDs and store them in the Audit schema for business review.
#
# Add audit columns to support data lineage and traceability.
#
# Create a reliable Bronze Delta table that will serve as the source for the Silver layer.
#
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
print(CATALOG)
print(BRONZE_EMPLOYEES)
print(AUDIT_DUPLICATE_EMPLOYEES)
print(EMPLOYEES_FILE)

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType,DataType,BooleanType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [0]:
#Employees schema
employees_schema = StructType([
    StructField("employee_id", StringType(), False),
    StructField("employee_name", StringType(), True),
    StructField("department_id", StringType(), True),
    StructField("job_title", StringType(), True),
    StructField("email", StringType(), True),
    StructField("hire_date", StringType(), True),
    StructField("is_active", BooleanType(), True)
])
# Read Employees master data from landing volume
bronze_employees_df = (spark.read
    .format("csv")
    .option("header", True)
    .schema(employees_schema)
    .load(EMPLOYEES_FILE)
)

#Source Data validation 
print(f"Total Records : {bronze_employees_df.count()}")

print("\nSchema:")
bronze_employees_df.printSchema()

print("\nColumns:")
print(bronze_employees_df.columns)

print("\nSampledata:")
display(bronze_employees_df.limit(10))

In [0]:
# Check the NULL and Blank employee_ids
null_blank_employee_id = bronze_employees_df.filter(col("employee_id").isNull() | (trim(col("employee_id")) == ""))

print(f"Total NULL or Blank employee_ids : {null_blank_employee_id.count()}")

display(null_blank_employee_id)

In [0]:
#Check the NULL and Blank employee name
null_blank_employee_name = bronze_employees_df.filter(col("employee_name").isNull() | (trim(col("employee_name")) == ""))

print(f"Total NULL or Blank employee_name : {null_blank_employee_name.count()}")

display(null_blank_employee_name)

In [0]:
#Check the NULL and Blank department_id
null_blank_department_id = bronze_employees_df.filter(col("department_id").isNull())

print(f"Total NULL or Blank department_id : {null_blank_department_id.count()}")

display(null_blank_department_id)


In [0]:
#Check the NULL and Blank job_title 
null_blank_job_title = bronze_employees_df.filter(col("job_title").isNull())

print(f"Total NULL or Blank job_title : {null_blank_job_title.count()}")

display(null_blank_job_title)


In [0]:
#Check the NULL and Blank hire_date 
null_blank_hire_date = bronze_employees_df.filter(col("hire_date").isNull())

print(f"Total NULL or Blank hire_date : {null_blank_hire_date.count()}")

display(null_blank_hire_date)


In [0]:
# ============================================================
# Identify Duplicate Employee Records
# Business Rule: Keep the first occurrence of each Employee ID and identify subsequent records as duplicates.
# ============================================================

window_spec = Window.partitionBy("employee_id").orderBy("employee_id")

employee_rank_df = (
    bronze_employees_df
        .withColumn(
            "row_num",
            row_number().over(window_spec)
        )
)

In [0]:
# ============================================================
# Retrieve Duplicate Employee Records
# ============================================================

duplicate_employees = (
    employee_rank_df
        .filter(col("row_num") > 1)
        .drop("row_num")
)

print(f"Duplicate employee Records : {duplicate_employees.count()}")

display(duplicate_employees)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

from pyspark.sql.functions import current_timestamp, lit

duplicate_employees = (
    duplicate_employees
        .withColumn("audit_timestamp", current_timestamp())
        .withColumn("source_table", lit("employees"))
        .withColumn("pipeline_layer", lit("Bronze"))
        .withColumn("issue_type", lit("Duplicate Record"))
)

display(duplicate_employees)

In [0]:
# ============================================================
# Write Duplicate Records to Audit Table
# ============================================================

duplicate_count = duplicate_employees.count()

if duplicate_count > 0:

    write_delta(
        df = duplicate_employees,
         table_name = AUDIT_DUPLICATE_EMPLOYEES
    )

    print(f"Successfully written {duplicate_count} duplicate record(s) to {AUDIT_DUPLICATE_EMPLOYEES}")

else:

    print("No duplicate Department records found. Audit table not created.")

In [0]:
# ============================================================
# Add Bronze Audit Columns
# ============================================================

bronze_employees_final_df = (
    bronze_employees_df
        .withColumn("load_timestamp", current_timestamp())
        .withColumn("source_file", lit("employees.csv"))
)

In [0]:
# ============================================================
# Write Bronze Delta Table
# ============================================================

write_delta(
    df=bronze_employees_final_df,
    table_name=BRONZE_EMPLOYEES
)

In [0]:
# ============================================================
# Validate Bronze Delta Table
# ============================================================

bronze_employees = spark.table(BRONZE_EMPLOYEES)

print(f"Total Bronze Records : {bronze_employees.count()}")

display(bronze_employees)

In [0]:
# ============================================================
# Bronze Employees complete summary
# ============================================================
print("=" * 60)
print("Bronze Employee Load Completed Successfully")
print("=" * 60)

print(f"Landing Records : {bronze_employees_df.count()}")

print(f"Audit Records : {duplicate_employees.count()}")

print(f"Bronze Records : {spark.table(BRONZE_EMPLOYEES).count()}")